In [1]:
import os
import json
import time
from pprint import pprint
from tqdm import tqdm

#avenieca sdk imports
from avenieca import Signal
from avenieca.producers import Event
from avenieca.api.model import *
from avenieca.api.eca import ECA
from dataclasses import dataclass, field
from typing import Optional, List

from config import *
import utils as util

In [4]:
@dataclass
class Pair():
    input: List[int] = field(default_factory=list)
    output: List[int] = field(default_factory=list)

In [5]:
data_path = os.getenv("ARC_FILES")

def get_train_or_test_split(file, mode="train"):
    ret = []
    with open(file) as f: 
        task = json.load(f)
        split = task[mode]
        for arc_pair in split:
            pair = Pair()
            input = []
            output = []
            for list in arc_pair['input']:    
                input.extend(list)
            for list in arc_pair['output']:
                output.extend(list)
            pair.input = input
            pair.output = output
            ret.append(pair)
    return ret
        
def create_train_and_test_pairs(file):
    train = get_train_or_test_split(file, "train")
    test = get_train_or_test_split(file, "test")
    return train, test

def prettyprint(res, status):
    try:
        pprint(res.__dict__)
    except:
        print(res)
    print(status)

In [6]:
arc_cell_broker_config = cell_config.broker_config
cell_event = Event(config=arc_cell_broker_config)

In [13]:
train, test = create_train_and_test_pairs('%s/0b148d64.json' % data_path)
print(len(train))
print(len(test))

3
1


In [14]:
print(train[0].input)

[8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 8, 8, 0, 0, 8, 0, 8, 0, 8, 8, 8, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 8, 8, 8, 8, 0, 0, 0, 8, 8, 8, 8, 0, 0, 0, 0, 8, 8, 0, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 0, 8, 8, 0, 0, 0, 0, 8, 8, 0, 0, 0, 8, 8, 8, 8, 8, 8, 0, 8, 8, 0, 8, 8, 0, 0, 0, 0, 8, 8, 8, 0, 8, 8, 8, 0, 0, 0, 8, 8, 0, 8, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 8, 8, 8, 8, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 8, 8, 8, 0, 8, 8, 8, 8, 0, 0, 8, 0, 0, 8, 8, 0, 8, 0, 0, 0, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 2, 2, 2, 2, 0, 0, 0, 0, 8, 8, 0, 8, 8, 0, 8, 2, 0, 2, 2, 2, 0, 0, 2, 2, 2, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 0, 0, 2, 2, 2, 2, 2, 2, 0, 2, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 8, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 8, 8, 0, 8, 8, 8, 0, 2, 2, 2, 2, 2, 2, 0, 2, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 

### Train

In [29]:
pair_index = 0
for i in tqdm(range(0, len(train[pair_index].input))):
    cell_signal = Signal(
        state=[float(train[pair_index].input[i])]
    )
    future = cell_event.publish(cell_signal)
    _ = future.get(timeout=60)
    time.sleep(0.2)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:52<00:00,  4.78it/s]


In [30]:
for i in tqdm(range(0, len(train[pair_index].output))):
    cell_signal = Signal(
        state=[float(train[pair_index].output[i])]
    )
    future = cell_event.publish(cell_signal)
    _ = future.get(timeout=60)
    time.sleep(0.2)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [00:52<00:00,  4.78it/s]


In [41]:
eca_server = os.getenv("ECA_SERVER")
config = Config(uri=eca_server, username=username, password=password)
eca = ECA(config)

In [ ]:
res, status = eca.ess.get_one(module_id="arc_cell",db_id=4)
prettyprint(res, status)

In [ ]:
res, status = eca.ess.get_one_sequence(module_id="arc_cell",sequence_id=500)
prettyprint(res, status)

### Test with REST API

In [32]:
eca_server = os.getenv("ECA_SERVER")
eca_secret = os.getenv("ECA_SECRET")

# ess_sequence = []
pair_index = 0
for i in tqdm(range(0, len(test[pair_index].input))):
    cell_ess: ESSResponse = util.create_ess_and_sequence(eca, [float(test[pair_index].input[i])], eca_secret, cell_config, None, False)
    # ess_sequence.append(cell_ess.id)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 297/297 [00:04<00:00, 66.47it/s]


In [33]:
previous_state = []
for i in range((len(ess_sequence)-1), -1, -1):
    previous_state.append(ess_sequence[i])

In [34]:
assert len(previous_state) == len(test[pair_index].output)

In [ ]:
print(ess_sequence)

In [ ]:
print(previous_state)

In [18]:
pair_index = 0
for i in tqdm(range(0, len(test[pair_index].output))):
    cell_ess: ESSResponse = util.create_ess_and_sequence(eca, [float(test[pair_index].output[i])], eca_secret, cell_config, None, False)
    

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 297/297 [00:04<00:00, 67.25it/s]


In [48]:
from datetime import datetime
start=datetime.now()

nsr = NextStateRequest(
    module_id="arc_cell",
    n=len(test[pair_index].output),
    range=50,
    recall=50,
    status='e',
    use_range_n_recall=False,
    current_state=previous_state[0],
    previous_state=previous_state
)
res, status = eca.cortex.predictions_raw(data=nsr)
# prettyprint(res, status)
print(datetime.now()-start)

Exception: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [36]:
assert len(test[pair_index].output) == len(res.next_state)

In [ ]:
print(res.next_state[0].list[0].state[0])

### Test with Stream

In [ ]:
for i in tqdm(range(0, len(test[pair_index].input))):
    cell_signal = Signal(
        state=[float(test[pair_index].input[i])]
    )
    future = cell_event.publish(cell_signal)
    _ = future.get(timeout=60)
    time.sleep(1)

In [ ]:
for i in tqdm(range(0, len(test[pair_index].output))):
    cell_signal = Signal(
        state=[float(test[pair_index].output[i])]
    )
    future = cell_event.publish(cell_signal)
    _ = future.get(timeout=60)
    time.sleep(73)

In [ ]:
with open('cell_236.txt') as f:
    log = [line.rstrip('\n') for line in f]

cell_log_n = [log[-1]]
for x in log[0:len(log)-1]:
    cell_log_n.append(x)
cell_log = cell_log_n

In [ ]:
print(len(cell_log))

In [ ]:
assert len(test[pair_index].output) == len(cell_log)

### Right shift

In [37]:
cell_log_n = [res.next_state[-1].list[0].state[0]]
for x in res.next_state[0:len(res.next_state)-1]:
    cell_log_n.append(x.list[0].state[0])
cell_log = cell_log_n

In [38]:
assert len(test[pair_index].output) == len(cell_log)

### Check accuracy

In [40]:
# REST API
accuracy_count = 0
for i in tqdm(range(0, len(test[pair_index].output))):
    print("%s, %s" % (test[pair_index].output[i], cell_log[i]))
    if float(test[pair_index].output[i]) == cell_log[i]:
        accuracy_count += 1
        
print(accuracy_count)
print((accuracy_count/len(test[pair_index].output)) * 100)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 297/297 [00:00<00:00, 155888.91it/s]

0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
4, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
0, 0.0
3, 0.0
0, 0.0
0, 0.0

In [ ]:
# Stream
accuracy_count = 0
for i in tqdm(range(0, len(test[pair_index].output))):
    # print("%s, %s" % test[pair_index].output[i], cell_log[i]))
    if float(test[pair_index].output[i]) == float(cell_log[i]):
        accuracy_count += 1
        
print(accuracy_count)
print((accuracy_count/len(test[pair_index].output)) * 100)